In [1]:
# 01_data_loading.ipynb

# %% [markdown]
# # Cross-Domain AI Governance Gap Analysis
# ## 01. Data Loading
#
# Load four datasets for the credit vs insurance cross-domain comparison.
# - [credit]    German Credit (UCI) — sensitive: sex, age
# - [credit]    Credit Card Approval Prediction (Kaggle) — sensitive: sex
#               Targets derived from credit_record.csv at two thresholds (30dpd, 60dpd).
# - [insurance] Medical Cost Personal (Kaggle) — sensitive: sex, age, smoker
# - [insurance] Porto Seguro Safe Driver (Kaggle) — indirect proxy variables

# %%
# Imports and path setup
import os
import urllib.request
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Paths are relative to the notebooks/ folder
DATA_DIR = os.path.join("..", "data")
RESULTS_DIR = os.path.join("..", "results")
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")

os.makedirs(DATA_DIR, exist_ok=True)

# %%
# Download German Credit from UCI if missing.
# The following Kaggle files must be placed in data/ manually:
#   application_record.csv, credit_record.csv  (Credit Card Approval Prediction)
#   insurance.csv                              (Medical Cost Personal)
#   porto_train.csv                            (Porto Seguro train.csv, unzipped & renamed)
german_file = os.path.join(DATA_DIR, "german.data")
if not os.path.exists(german_file):
    urllib.request.urlretrieve(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
        german_file,
    )

# %%
# Load German Credit (Statlog) — UCI [credit domain]
# https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data
german_cols = [
    "status_checking", "duration", "credit_history", "purpose", "credit_amount",
    "savings", "employment", "installment_rate", "personal_status_sex", "other_debtors",
    "residence_since", "property", "age", "other_installment", "housing",
    "existing_credits", "job", "num_dependents", "telephone", "foreign_worker", "target"
]

df_german = pd.read_csv(german_file, sep=r"\s+", header=None, names=german_cols)

# Recode target: 1 = good, 2 = bad -> 1 = bad (default), 0 = good
df_german["target"] = (df_german["target"] == 2).astype(int)

sex_map = {"A91": "male", "A92": "female", "A93": "male", "A94": "male", "A95": "female"}
df_german["sex"] = df_german["personal_status_sex"].map(sex_map)
df_german["age_group"] = np.where(df_german["age"] >= 25, "old", "young")
df_german["domain"] = "credit"

print("German Credit shape:", df_german.shape)
df_german.head()

# %%
# Build targets from Credit Card Approval Prediction — Kaggle [credit domain]
# Two files: application_record.csv (features), credit_record.csv (monthly STATUS).
# STATUS codes: 0=1-29d, 1=30-59d, 2=60-89d, 3=90-119d, 4=120-149d, 5=150+d/writeoff,
#               C=paid off, X=no loan.
# Two labels are built at different delinquency thresholds:
#   target_30dpd : bad if any month reaches 30+ days past due  (STATUS in {1,2,3,4,5})
#   target_60dpd : bad if any month reaches 60+ days past due  (STATUS in {2,3,4,5})
PRIMARY_TARGET = "target_30dpd"  # which threshold column is used as the default "target"

df_app = pd.read_csv(os.path.join(DATA_DIR, "application_record.csv"))
df_credit = pd.read_csv(os.path.join(DATA_DIR, "credit_record.csv"))

status_num = df_credit["STATUS"].replace({"C": -1, "X": -1}).astype(int)
df_credit = df_credit.assign(status_num=status_num)

worst = df_credit.groupby("ID")["status_num"].max()

labels = pd.DataFrame({
    "target_30dpd": (worst >= 1).astype(int),
    "target_60dpd": (worst >= 2).astype(int),
})

df_home = df_app.merge(labels, on="ID", how="inner")
df_home["target"] = df_home[PRIMARY_TARGET]

df_home["sex"] = df_home["CODE_GENDER"].map({"M": "male", "F": "female"})
df_home = df_home[df_home["sex"].notna()].copy()
df_home["domain"] = "credit"

print("Credit Card Approval shape:", df_home.shape)
print("Primary target:", PRIMARY_TARGET)
print("target_30dpd pos_rate:", round(df_home["target_30dpd"].mean(), 3))
print("target_60dpd pos_rate:", round(df_home["target_60dpd"].mean(), 3))
df_home[["ID", "target", "target_30dpd", "target_60dpd", "sex", "AMT_INCOME_TOTAL"]].head()

# %%
# Load Medical Cost Personal — Kaggle [insurance domain]
# Columns: age, sex, bmi, children, smoker, region, charges.
# Original target is continuous (charges); convert to an underwriting-style binary
# target by labeling the high-cost quantile as 1.
# https://www.kaggle.com/datasets/mirichoi0218/insurance
df_ins = pd.read_csv(os.path.join(DATA_DIR, "insurance.csv"))

# Sensitive attributes: sex, age, smoker
df_ins["sex"] = df_ins["sex"].str.lower()
df_ins["age_group"] = np.where(df_ins["age"] >= 40, "old", "young")
df_ins["smoker_flag"] = (df_ins["smoker"] == "yes").astype(int)

# Binary target: top 25% of charges = high risk (threshold can be tuned in experiments)
threshold = df_ins["charges"].quantile(0.75)
df_ins["target"] = (df_ins["charges"] >= threshold).astype(int)
df_ins["domain"] = "insurance"

print("Medical Cost shape:", df_ins.shape)
print("High-risk threshold (charges 75th pct):", round(threshold, 2))
print(df_ins["target"].value_counts(normalize=True).round(3))
df_ins.head()

# %%
# Load Porto Seguro Safe Driver Prediction — Kaggle [insurance domain]
# No explicit sensitive attribute; use anonymized individual-level proxy variables (ps_ind_*)
# target: 1 = claim filed next year
# https://www.kaggle.com/competitions/porto-seguro-safe-driver-prediction
df_porto = pd.read_csv(os.path.join(DATA_DIR, "porto_train.csv"))

# Replace the -1 missing indicator with NaN
df_porto = df_porto.replace(-1, np.nan)
df_porto["domain"] = "insurance"

# Proxy candidates: individual-characteristic categorical/binary features
proxy_cols = [c for c in df_porto.columns if c.startswith("ps_ind") and ("cat" in c or "bin" in c)]

print("Porto Seguro shape:", df_porto.shape)
print("target dist:", df_porto["target"].value_counts(normalize=True).round(4).to_dict())
print("Proxy candidates:", proxy_cols)
df_porto.head()

# %%
# Build cross-domain summary table
summary = pd.DataFrame([
    {"domain": "credit", "dataset": "German Credit", "n_rows": len(df_german),
     "pos_rate": round(df_german["target"].mean(), 3), "sensitive": "sex, age_group"},
    {"domain": "credit", "dataset": "Credit Card Approval (30dpd)", "n_rows": len(df_home),
     "pos_rate": round(df_home["target_30dpd"].mean(), 3), "sensitive": "sex"},
    {"domain": "credit", "dataset": "Credit Card Approval (60dpd)", "n_rows": len(df_home),
     "pos_rate": round(df_home["target_60dpd"].mean(), 3), "sensitive": "sex"},
    {"domain": "insurance", "dataset": "Medical Cost", "n_rows": len(df_ins),
     "pos_rate": round(df_ins["target"].mean(), 3), "sensitive": "sex, age_group, smoker"},
    {"domain": "insurance", "dataset": "Porto Seguro", "n_rows": len(df_porto),
     "pos_rate": round(df_porto["target"].mean(), 3), "sensitive": "proxy (ps_ind_*)"},
])
print(summary.to_string(index=False))

os.makedirs(TAB_DIR, exist_ok=True)
summary.to_csv(os.path.join(TAB_DIR, "01_crossdomain_summary.csv"), index=False)

# %%
# Persist intermediate frames for reuse in the next notebook
os.makedirs(os.path.join(DATA_DIR, "processed"), exist_ok=True)
df_german.to_parquet(os.path.join(DATA_DIR, "processed", "german.parquet"))
df_home.to_parquet(os.path.join(DATA_DIR, "processed", "credit_card_approval.parquet"))
df_ins.to_parquet(os.path.join(DATA_DIR, "processed", "insurance.parquet"))
df_porto.to_parquet(os.path.join(DATA_DIR, "processed", "porto_seguro.parquet"))

German Credit shape: (1000, 24)
Credit Card Approval shape: (36457, 23)
Primary target: target_30dpd
target_30dpd pos_rate: 0.118
target_60dpd pos_rate: 0.017
Medical Cost shape: (1338, 11)
High-risk threshold (charges 75th pct): 16639.91
target
0    0.75
1    0.25
Name: proportion, dtype: float64
Porto Seguro shape: (595212, 60)
target dist: {0: 0.9636, 1: 0.0364}
Proxy candidates: ['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_ind_06_bin', 'ps_ind_07_bin', 'ps_ind_08_bin', 'ps_ind_09_bin', 'ps_ind_10_bin', 'ps_ind_11_bin', 'ps_ind_12_bin', 'ps_ind_13_bin', 'ps_ind_16_bin', 'ps_ind_17_bin', 'ps_ind_18_bin']
   domain                      dataset  n_rows  pos_rate              sensitive
   credit                German Credit    1000     0.300         sex, age_group
   credit Credit Card Approval (30dpd)   36457     0.118                    sex
   credit Credit Card Approval (60dpd)   36457     0.017                    sex
insurance                 Medical Cost    1338     0.25